In [1]:
import zipfile

with zipfile.ZipFile('Q2.zip', 'r') as zip_ref:
    zip_ref.extractall()

In [2]:
import os
print(os.listdir())

['.config', 'Q2.zip', 'Q2', '__MACOSX', 'sample_data']


In [3]:
print(os.listdir("Q2"))

['answers.txt', 'webpages', '.DS_Store', 'actions.txt']


In [4]:
print(os.listdir("Q2/webpages")[:5])

['stack_datastructure_wiki', 'references', 'stack_oracle', 'stackoverflow', 'stackmagazine']


In [5]:
import os
import re
import math

stop_words = {
    "a", "an", "the", "they", "these", "this", "for",
    "is", "are", "was", "of", "or", "and", "does",
    "will", "whose"
}

In [6]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[{}()\[\].,;\'"?!:#\-]', ' ', text)
    words = text.split()
    return [w for w in words if w not in stop_words]

In [7]:
class MySet:
    def __init__(self):
        self.data = set()

    def addElement(self, element):
        self.data.add(element)

    def union(self, otherSet):
        result = MySet()
        result.data = self.data.union(otherSet.data)
        return result

    def intersection(self, otherSet):
        result = MySet()
        result.data = self.data.intersection(otherSet.data)
        return result

In [8]:
class Position:
    def __init__(self, page, index):
        self.page = page
        self.index = index

    def getPageEntry(self):
        return self.page

    def getWordIndex(self):
        return self.index


In [9]:
class WordEntry:
    def __init__(self, word):
        self.word = word
        self.positions = []

    def addPosition(self, position):
        self.positions.append(position)

    def addPositions(self, positions):
        self.positions.extend(positions)

    def getAllPositionsForThisWord(self):
        return self.positions

    def getTermFrequency(self, page):
        count = 0
        for pos in self.positions:
            if pos.getPageEntry().pageName == page.pageName:
                count += 1
        return count

In [10]:
class PageIndex:
    def __init__(self):
        self.wordEntries = {}

    def addPositionForWord(self, word, position):
        if word not in self.wordEntries:
            self.wordEntries[word] = WordEntry(word)
        self.wordEntries[word].addPosition(position)

    def getWordEntries(self):
        return self.wordEntries

In [11]:
class PageEntry:
    def __init__(self, pageName):
        self.pageName = pageName
        self.pageIndex = PageIndex()
        self.words = []
        self.read_page()

    def read_page(self):
        with open(f"Q2/webpages/{self.pageName}", 'r') as f:
            text = f.read()
            cleaned_words = clean_text(text)
            self.words = cleaned_words

            for i, word in enumerate(cleaned_words):
                pos = Position(self, i + 1)
                self.pageIndex.addPositionForWord(word, pos)

    def getPageIndex(self):
        return self.pageIndex

    def getWordCount(self):
        return len(self.words)

In [12]:
class MyHashTable:
    def __init__(self):
        self.table = {}

    def addPositionsForWord(self, wordEntry):
        word = wordEntry.word
        if word not in self.table:
            self.table[word] = wordEntry
        else:
            self.table[word].addPositions(wordEntry.positions)

    def getWordEntry(self, word):
        return self.table.get(word, None)

In [13]:
class InvertedPageIndex:
    def __init__(self):
        self.hashTable = MyHashTable()
        self.pages = []

    def addPage(self, page):
        self.pages.append(page)
        wordEntries = page.getPageIndex().getWordEntries()

        for wordEntry in wordEntries.values():
            self.hashTable.addPositionsForWord(wordEntry)

    def getPagesWhichContainWord(self, word):
        entry = self.hashTable.getWordEntry(word)
        if entry is None:
            return None

        pages = set()
        for pos in entry.getAllPositionsForThisWord():
            pages.add(pos.getPageEntry())
        return pages

In [15]:
def compute_tf(word, page):
    wordEntries = page.getPageIndex().getWordEntries()
    if word not in wordEntries:
        return 0
    return wordEntries[word].getTermFrequency(page) / page.getWordCount()


def compute_idf(word, index):
    total_pages = len(index.pages)
    pages_with_word = index.getPagesWhichContainWord(word)

    if pages_with_word is None:
        return 0

    return math.log(total_pages / len(pages_with_word))


def compute_tfidf(word, page, index):
    return compute_tf(word, page) * compute_idf(word, index)

In [18]:
class SearchEngine:
    def __init__(self):
        self.index = InvertedPageIndex()
        self.pages = {}

    def performAction(self, action):
        parts = action.strip().split()

        if parts[0] == "addPage":
            pageName = parts[1]
            page = PageEntry(pageName)
            self.pages[pageName] = page
            self.index.addPage(page)

        elif parts[0] == "queryFindPagesWhichContainWord":
            word = parts[1].lower()
            pages = self.index.getPagesWhichContainWord(word)

            if pages is None:
                print(f"No webpage contains word {word}")
            else:
                print(", ".join(sorted([p.pageName for p in pages])))
        elif parts[0] == "queryFindPositionsOfWordInAPage":
            word = parts[1].lower()
            pageName = parts[2]

            if pageName not in self.pages:
                print(f"No webpage {pageName} found")
                return

            page = self.pages[pageName]
            wordEntries = page.getPageIndex().getWordEntries()

            if word not in wordEntries:
                print(f"Webpage {pageName} does not contain word {word}")
            else:
                positions = [str(pos.getWordIndex()) for pos in wordEntries[word].positions]
                print(", ".join(positions))

        elif parts[0] == "queryFindPagesWhichContainAllWords":
            words = [w.lower() for w in parts[1:]]
            scores = {}

            for page in self.index.pages:
                score = 0
                for word in words:
                    score += compute_tfidf(word, page, self.index)

                if score > 0:
                    scores[page.pageName] = score

            sorted_pages = sorted(scores.items(), key=lambda x: x[1], reverse=True)

            print(", ".join([p for p, _ in sorted_pages]))

In [19]:
engine = SearchEngine()

with open("Q2/actions.txt", "r") as f:
    actions = f.readlines()

for action in actions:
    engine.performAction(action)

No webpage contains word delhi
stack_datastructure_wiki
stack_datastructure_wiki
Webpage stack_datastructure_wiki does not contain word magazines
No webpage contains word allain
stack_cprogramming
stack_cprogramming
stack_cprogramming
stack_oracle
stack_cprogramming, stack_datastructure_wiki, stackoverflow
stackmagazine
